In [1]:
import pandas as pd

df = pd.read_csv("../data/final_mental_health_dataset.csv")
df.head()

,text,stress_level,emotion
0,Welcome to /r/depression's check-in post - a p...,High,Unknown
1,We understand that most people who reply immed...,High,Unknown
2,"I've always wanted to go back to school, the c...",High,Unknown
3,All I wan is to be loved. I just want someone...,High,Unknown
4,"I don’t even drive, didn’t finish my high scho...",High,Unknown


In [2]:
emotion_df = df[df["emotion"] != "Unknown"].copy()
emotion_df["emotion"].value_counts()

emotion
Happy       6760
Sad         5796
Angry       2708
Fear        2372
Love        1641
Surprise     719
Name: count, dtype: int64

In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
emotion_df["clean_text"] = emotion_df["text"].apply(clean_text)

In [6]:
from sklearn.model_selection import train_test_split

X = emotion_df["clean_text"]
y = emotion_df["emotion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_emotion = TfidfVectorizer(max_features=12000, ngram_range=(1,2))

X_train_tfidf = tfidf_emotion.fit_transform(X_train)
X_test_tfidf  = tfidf_emotion.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

emotion_model = LogisticRegression(max_iter=1000)

emotion_model.fit(X_train_tfidf, y_train)

y_pred = emotion_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       Angry       0.93      0.82      0.87       542
        Fear       0.89      0.79      0.84       475
       Happy       0.85      0.97      0.91      1352
        Love       0.87      0.69      0.77       328
         Sad       0.91      0.95      0.93      1159
    Surprise       0.86      0.50      0.63       144

    accuracy                           0.88      4000
   macro avg       0.88      0.79      0.82      4000
weighted avg       0.89      0.88      0.88      4000



In [9]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/emotion_model.pkl", "wb") as f:
    pickle.dump(emotion_model, f)

with open("../models/tfidf_emotion.pkl", "wb") as f:
    pickle.dump(tfidf_emotion, f)

print("Emotion model saved.")


Emotion model saved.
